installing required libraries

In [ ]:

!pip install torch torchvision torchaudio --quiet
!pip install opencv-python numpy pillow tqdm matplotlib gradio --quiet


create working directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content


/content


In [ ]:
# link Python source files
!ln -sf /content/drive/MyDrive/brain_tumor_project/src/unet_model.py .
!ln -sf /content/drive/MyDrive/brain_tumor_project/src/brisc_dataset.py .
!ln -sf /content/drive/MyDrive/brain_tumor_project/src/train_unet.py .
!ln -sf /content/drive/MyDrive/brain_tumor_project/src/inference_app.py .

# link trained model (if exists)
!ln -sf /content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth .


In [ ]:
!ls


brisc_dataset.py  inference_app.py  train_unet.py   unet_model.py
drive		  sample_data	    unet_brisc.pth


In [ ]:
!nvidia-smi


Mon Nov 17 01:42:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!ls /content/drive/MyDrive/brain_tumor_project/data/train/images | head
!ls /content/drive/MyDrive/brain_tumor_project/data/train/masks | head


brisc2025_train_00001_gl_ax_t1.jpg
brisc2025_train_00002_gl_ax_t1.jpg
brisc2025_train_00003_gl_ax_t1.jpg
brisc2025_train_00004_gl_ax_t1.jpg
brisc2025_train_00005_gl_ax_t1.jpg
brisc2025_train_00006_gl_ax_t1.jpg
brisc2025_train_00007_gl_ax_t1.jpg
brisc2025_train_00008_gl_ax_t1.jpg
brisc2025_train_00009_gl_ax_t1.jpg
brisc2025_train_00010_gl_ax_t1.jpg
brisc2025_train_00001_gl_ax_t1.png
brisc2025_train_00002_gl_ax_t1.png
brisc2025_train_00003_gl_ax_t1.png
brisc2025_train_00004_gl_ax_t1.png
brisc2025_train_00005_gl_ax_t1.png
brisc2025_train_00006_gl_ax_t1.png
brisc2025_train_00007_gl_ax_t1.png
brisc2025_train_00008_gl_ax_t1.png
brisc2025_train_00009_gl_ax_t1.png
brisc2025_train_00010_gl_ax_t1.png


In [ ]:
!ls /content/drive/MyDrive/brain_tumor_project/data/test/images | head
!ls /content/drive/MyDrive/brain_tumor_project/data/test/masks | head


brisc2025_test_00001_gl_ax_t1.jpg
brisc2025_test_00002_gl_ax_t1.jpg
brisc2025_test_00003_gl_ax_t1.jpg
brisc2025_test_00004_gl_ax_t1.jpg
brisc2025_test_00005_gl_ax_t1.jpg
brisc2025_test_00006_gl_ax_t1.jpg
brisc2025_test_00007_gl_ax_t1.jpg
brisc2025_test_00008_gl_ax_t1.jpg
brisc2025_test_00009_gl_ax_t1.jpg
brisc2025_test_00010_gl_ax_t1.jpg
brisc2025_test_00001_gl_ax_t1.png
brisc2025_test_00002_gl_ax_t1.png
brisc2025_test_00003_gl_ax_t1.png
brisc2025_test_00004_gl_ax_t1.png
brisc2025_test_00005_gl_ax_t1.png
brisc2025_test_00006_gl_ax_t1.png
brisc2025_test_00007_gl_ax_t1.png
brisc2025_test_00008_gl_ax_t1.png
brisc2025_test_00009_gl_ax_t1.png
brisc2025_test_00010_gl_ax_t1.png


In [ ]:
%%writefile /content/drive/MyDrive/brain_tumor_project/src/unet_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.pools = nn.ModuleList()

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            self.pools.append(nn.MaxPool2d(2))
            in_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        self.ups = nn.ModuleList()
        self.up_convs = nn.ModuleList()

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.up_convs.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down, pool in zip(self.downs, self.pools):
            x = down(x)
            skip_connections.append(x)
            x = pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(len(self.ups)):
            x = self.ups[i](x)
            skip = skip_connections[i]

            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])

            x = torch.cat((skip, x), dim=1)
            x = self.up_convs[i](x)

        return self.final_conv(x)


Overwriting /content/drive/MyDrive/brain_tumor_project/src/unet_model.py


In [ ]:
%%writefile /content/drive/MyDrive/brain_tumor_project/src/brisc_dataset.py
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class BRISCDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=256):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.images = sorted(os.listdir(img_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = img_name.replace(".jpg", ".png")

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, (self.img_size, self.img_size))
        mask = cv2.resize(mask, (self.img_size, self.img_size))

        img = img.astype(np.float32)/255.0
        mask = (mask > 127).astype(np.float32)

        img = np.expand_dims(img, axis=0)
        mask = np.expand_dims(mask, axis=0)

        return torch.tensor(img, dtype=torch.float32), torch.tensor(mask, dtype=torch.float32)


Overwriting /content/drive/MyDrive/brain_tumor_project/src/brisc_dataset.py


In [ ]:
%%writefile /content/drive/MyDrive/brain_tumor_project/src/train_unet.py
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import os

from brisc_dataset import BRISCDataset
from unet_model import UNet

def dice(pred, target, eps=1e-6):
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    inter = (pred * target).sum()
    return (2*inter + eps)/(pred.sum() + target.sum() + eps)

def train():
    TRAIN_IMG = "/content/train_data/images"
    TRAIN_MASK = "/content/train_data/masks"


    FINAL_MODEL = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"
    CHECKPOINT_DIR = "/content/drive/MyDrive/brain_tumor_project/checkpoints"

    dataset = BRISCDataset(TRAIN_IMG, TRAIN_MASK)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=8,shuffle=True, num_workers=2, pin_memory=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Training on:", device)

    model = UNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()

    best_dice = 0

    for epoch in range(1, 11):
        model.train()
        train_loss = 0

        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/10"):
            imgs, masks = imgs.to(device), masks.to(device)

            logits = model(imgs)
            loss = loss_fn(logits, masks)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_loss += loss.item()

        # validation
        model.eval()
        dices = []

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)

                logits = model(imgs)
                preds = (torch.sigmoid(logits) > 0.5).float()

                dices.append(dice(preds, masks).item())

        avg_dice = sum(dices)/len(dices)
        print(f"Epoch {epoch}: Loss={train_loss:.4f}, Dice={avg_dice:.4f}")

        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.state_dict(), FINAL_MODEL)
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)
            torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/epoch_{epoch}_dice_{avg_dice:.4f}.pth")
            print("🔥 New best model saved!")

if __name__ == "__main__":
    train()


Overwriting /content/drive/MyDrive/brain_tumor_project/src/train_unet.py


In [ ]:
%%writefile /content/drive/MyDrive/brain_tumor_project/src/inference_app.py
import cv2
import numpy as np
import torch
import gradio as gr
from unet_model import UNet

MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

def predict_mask(image):
    if len(image.shape) == 2:
        gray = image
    else:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape

    resized = cv2.resize(gray, (256, 256))
    normalized = resized.astype(np.float32)/255.0

    tensor = torch.tensor(normalized, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(tensor)
        mask = (torch.sigmoid(logits) > 0.5).float().cpu().numpy()[0,0]

    mask = cv2.resize(mask, (w, h))
    mask_uint8 = (mask * 255).astype(np.uint8)

    overlay = image.copy()
    overlay[mask_uint8 > 127] = [255, 0, 0]
    blended = cv2.addWeighted(image, 0.6, overlay, 0.4, 0)

    return blended, mask_uint8

iface = gr.Interface(
    fn=predict_mask,
    inputs=gr.Image(type="numpy", label="Upload MRI Image"),
    outputs=[
        gr.Image(type="numpy", label="Overlay Prediction"),
        gr.Image(type="numpy", label="Segmentation Mask")
    ],
    title="BRISC Brain Tumor Segmentation (U-Net)"
)

if __name__ == "__main__":
    iface.launch(share=True)


Overwriting /content/drive/MyDrive/brain_tumor_project/src/inference_app.py


In [ ]:
!python train_unet.py


python3: can't open file '/content/drive/MyDrive/brain_tumor_project/train_unet.py': [Errno 2] No such file or directory


STEP 1 — Create an Evaluation Script (evaluate_unet.py)

In [ ]:
%%writefile /content/evaluate_unet.py
import os
import cv2
import numpy as np
import torch
from tqdm import tqdm
from unet_model import UNet

# ---- PATHS ----
TEST_IMG = "/content/drive/MyDrive/brain_tumor_project/data/test/images"
TEST_MASK = "/content/drive/MyDrive/brain_tumor_project/data/test/masks"
MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- METRIC FUNCTIONS ----

def dice_score(pred, gt, eps=1e-6):
    pred = pred.flatten()
    gt = gt.flatten()
    intersection = (pred * gt).sum()
    return (2 * intersection + eps) / (pred.sum() + gt.sum() + eps)

def iou_score(pred, gt, eps=1e-6):
    pred = pred.flatten()
    gt = gt.flatten()
    intersection = (pred * gt).sum()
    union = pred.sum() + gt.sum() - intersection
    return (intersection + eps) / (union + eps)

def precision(pred, gt, eps=1e-6):
    pred = pred.flatten()
    gt = gt.flatten()
    tp = (pred * gt).sum()
    fp = pred.sum() - tp
    return (tp + eps) / (tp + fp + eps)

def recall(pred, gt, eps=1e-6):
    pred = pred.flatten()
    gt = gt.flatten()
    tp = (pred * gt).sum()
    fn = gt.sum() - tp
    return (tp + eps) / (tp + fn + eps)

def accuracy(pred, gt):
    pred = pred.flatten()
    gt = gt.flatten()
    return (pred == gt).mean()

# ---- LOAD MODEL ----
model = UNet()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

dice_list, iou_list, prec_list, rec_list, acc_list = [], [], [], [], []

# ---- EVALUATION LOOP ----
print("\nEvaluating on BRISC test set...\n")

for fname in tqdm(sorted(os.listdir(TEST_IMG))):
    img_path = os.path.join(TEST_IMG, fname)
    mask_path = os.path.join(TEST_MASK, fname.replace(".jpg", ".png"))

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    h, w = img.shape
    img_r = cv2.resize(img, (256, 256)) / 255.0
    gt_r = (cv2.resize(gt_mask, (256, 256)) > 127).astype(np.float32)

    img_tensor = torch.tensor(img_r).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        pred = model(img_tensor)
        pred = (torch.sigmoid(pred) > 0.5).float().cpu().numpy()[0, 0]

    # ---- Compute Metrics ----
    dice_list.append(dice_score(pred, gt_r))
    iou_list.append(iou_score(pred, gt_r))
    prec_list.append(precision(pred, gt_r))
    rec_list.append(recall(pred, gt_r))
    acc_list.append(accuracy(pred, gt_r))

# ---- Print final results ----
print("\n============== Final Test Metrics ==============\n")
print("Average Dice Score:      ", sum(dice_list)/len(dice_list))
print("Average IoU Score:       ", sum(iou_list)/len(iou_list))
print("Average Precision:        ", sum(prec_list)/len(prec_list))
print("Average Recall:           ", sum(rec_list)/len(rec_list))
print("Average Accuracy:         ", sum(acc_list)/len(acc_list))
print("\n================================================\n")


Writing /content/evaluate_unet.py


In [ ]:
!python /content/evaluate_unet.py



Evaluating on BRISC test set...

  1% 5/860 [00:06<18:05,  1.27s/it]
Traceback (most recent call last):
  File "/content/evaluate_unet.py", line 65, in <module>
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C


In [ ]:
# Create Visualization Script
%%writefile /content/visualize_test_predictions.py
import sys, os
sys.path.append("/content/drive/MyDrive/brain_tumor_project/src")

import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from unet_model import UNet

TEST_IMG = "/content/drive/MyDrive/brain_tumor_project/data/test/images"
TEST_MASK = "/content/drive/MyDrive/brain_tumor_project/data/test/masks"
MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

files = sorted(os.listdir(TEST_IMG))

# visualize 10 random samples
idxs = np.random.choice(len(files), 10, replace=False)

for idx in idxs:
    fname = files[idx]
    img_path = os.path.join(TEST_IMG, fname)
    mask_path = os.path.join(TEST_MASK, fname.replace(".jpg", ".png"))

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    h, w = img.shape

    img_r = cv2.resize(img, (256,256))/255.0
    img_tensor = torch.tensor(img_r).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        pred = model(img_tensor)
        pred = (torch.sigmoid(pred) > 0.5).float().cpu().numpy()[0,0]

    pred_up = cv2.resize(pred, (w,h))

    # Overlay
    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    overlay[pred_up > 0.5] = [255, 0, 0]

    # plotting
    plt.figure(figsize=(12,4))

    plt.subplot(1,4,1)
    plt.imshow(img, cmap="gray")
    plt.title("MRI Image")
    plt.axis("off")

    plt.subplot(1,4,2)
    plt.imshow(gt, cmap="gray")
    plt.title("Ground Truth Mask")
    plt.axis("off")

    plt.subplot(1,4,3)
    plt.imshow(pred_up, cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1,4,4)
    plt.imshow(overlay)
    plt.title("Overlay")
    plt.axis("off")

    plt.show()


Overwriting /content/visualize_test_predictions.py


In [ ]:
!python /content/visualize_test_predictions.py


Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)
Figure(1200x400)


***** CREATE A CUSTOM UI/UX APP FOR USER *****

In [ ]:
%%writefile /content/web_app.py
import sys, os
sys.path.append("/content/drive/MyDrive/brain_tumor_project/src")

import gradio as gr
import torch
import cv2
import numpy as np
from unet_model import UNet

MODEL = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load model once
model = UNet()
model.load_state_dict(torch.load(MODEL, map_location=DEVICE))
model.to(DEVICE)
model.eval()

def segment(image, color_choice, opacity):
    if image is None:
        return None, None, "No image uploaded"

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    # Preprocess
    img_r = cv2.resize(gray, (256,256)) / 255.0
    img_tensor = torch.tensor(img_r).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    # Predict
    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()[0,0]
        pred = (probs > 0.5).astype(np.uint8)

    # Resize prediction
    pred_mask = cv2.resize(pred, (w,h))
    prob_map = cv2.resize(probs, (w,h))

    # Calculate Tumor Probability
    tumor_prob = float(prob_map[pred_mask == 1].mean() * 100) if pred_mask.sum() > 0 else 0.0

    # Choose color
    colors = {
        "Red": (255, 0, 0),
        "Green": (0, 255, 0),
        "Blue": (0, 0, 255),
        "Yellow": (255, 255, 0),
        "Cyan": (0, 255, 255)
    }
    color = colors[color_choice]

    # Overlay
    overlay = image.copy()
    overlay[pred_mask > 0] = color

    # Apply opacity
    blended = cv2.addWeighted(image, 1-opacity, overlay, opacity, 0)

    # Prepare mask as 0/255 image
    mask_uint8 = (pred_mask * 255).astype(np.uint8)

    return blended, mask_uint8, f"{tumor_prob:.2f}% tumor confidence"

# Custom CSS
custom_css = """
body {background-color: #0B0C10;}
.gradio-container {background-color: #1F2833 !important;}
h1 {color: #66FCF1 !important; text-align:center !important; font-size: 2.3rem !important;}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as app:
    gr.Markdown("<h1>🧠 Advanced Brain Tumor Segmentation</h1>")
    gr.Markdown("<p style='text-align:center; color:#C5C6C7'>Upload an MRI scan & customize tumor visualization.</p>")

    with gr.Row():
        image_input = gr.Image(label="Upload MRI Image", type="numpy")

    with gr.Row():
        color_choice = gr.Dropdown(
            ["Red", "Green", "Blue", "Yellow", "Cyan"],
            value="Red",
            label="Tumor Color"
        )
        opacity_slider = gr.Slider(
            0.1, 1.0, value=0.6,
            label="Mask Opacity"
        )

    segment_btn = gr.Button("Run Segmentation", variant="primary")

    with gr.Row():
        output_overlay = gr.Image(label="Overlay Prediction")
        output_mask = gr.Image(label="Binary Mask")

    tumor_probability = gr.Textbox(label="Tumor Probability Score")

    segment_btn.click(
        segment,
        inputs=[image_input, color_choice, opacity_slider],
        outputs=[output_overlay, output_mask, tumor_probability]
    )

app.launch(share=True)


Writing /content/web_app.py


In [ ]:
!python /content/web_app.py


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jsonschema/_format.py", line 304, in <module>
    import rfc3987
ModuleNotFoundError: No module named 'rfc3987'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/web_app.py", line 4, in <module>
    import gradio as gr
  File "/usr/local/lib/python3.12/dist-packages/gradio/__init__.py", line 8, in <module>
    from gradio import components, layouts, mcp, themes
  File "/usr/local/lib/python3.12/dist-packages/gradio/mcp.py", line 46, in <module>
    class GradioMCPServer:
  File "/usr/local/lib/python3.12/dist-packages/gradio/mcp.py", line 59, in GradioMCPServer
    from mcp import types
  File "/usr/local/lib/python3.12/dist-packages/mcp/__init__.py", line 4, in <module>
    from .server.session import ServerSession
  File "/usr/local/lib/python3.12/dist-packages/mcp/server/__init__.py", line 1, in <module>
    from .fastmcp import F

this is a combine app for both the models to visulize the results


In [ ]:
%%writefile src/brain_tumor_demo_app.py
%cd /content/drive/MyDrive/brain_tumor_project

import sys, os
sys.path.append("/content/drive/MyDrive/brain_tumor_project/src")

import cv2
import numpy as np
import torch
import gradio as gr

from unet_model import UNet

from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

# -----------------------------
# Paths & device
# -----------------------------
UNET_MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"
RCNN_MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/rcnn/output/model_final.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# -----------------------------
# Load U-Net model
# -----------------------------
unet = UNet().to(DEVICE)
unet.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
unet.eval()
print("Loaded U-Net weights.")

# -----------------------------
# Load Mask R-CNN (Detectron2)
# -----------------------------
cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # single class: tumor
cfg.MODEL.WEIGHTS = RCNN_MODEL_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.MODEL.DEVICE = DEVICE

predictor_rcnn = DefaultPredictor(cfg)
print("Loaded Mask R-CNN weights.")

# -----------------------------
# Helper: U-Net prediction
# -----------------------------
def run_unet(image_rgb: np.ndarray):
    """
    image_rgb: H x W x 3, RGB (from Gradio)
    Returns: overlay RGB, mask uint8, confidence float
    """
    h, w, _ = image_rgb.shape

    # Grayscale + resize to 256x256 (as in training)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (256, 256))
    normalized = resized.astype(np.float32) / 255.0

    tensor = torch.from_numpy(normalized).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = unet(tensor)
        probs = torch.sigmoid(logits)[0, 0]  # (256, 256)

    mask_small = probs.cpu().numpy()
    # confidence: max probability in predicted mask
    confidence = float(mask_small.max())

    # Resize back to original image size
    mask_full = cv2.resize(mask_small, (w, h))
    binary = mask_full > 0.5

    overlay = image_rgb.copy()
    # Paint tumor region in red
    overlay[binary] = [255, 0, 0]
    blended = cv2.addWeighted(image_rgb, 0.7, overlay, 0.3, 0)

    mask_uint8 = (binary.astype(np.uint8) * 255)

    return blended, mask_uint8, confidence

# -----------------------------
# Helper: Mask R-CNN prediction
# -----------------------------
def run_rcnn(image_rgb: np.ndarray):
    """
    image_rgb: H x W x 3, RGB
    Returns: overlay RGB, mask uint8, confidence float
    """
    # Detectron2 expects BGR
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    outputs = predictor_rcnn(image_bgr)
    instances = outputs["instances"].to("cpu")

    h, w, _ = image_rgb.shape

    if len(instances) == 0:
        empty = np.zeros((h, w), dtype=np.uint8)
        return image_rgb, empty, 0.0

    masks = instances.pred_masks.numpy()  # (N, H, W)
    scores = instances.scores.numpy()

    best_idx = scores.argmax()
    best_mask = masks[best_idx]
    confidence = float(scores[best_idx])

    overlay = image_rgb.copy()
    # Paint tumor region in green
    overlay[best_mask] = [0, 255, 0]
    blended = cv2.addWeighted(image_rgb, 0.7, overlay, 0.3, 0)

    mask_uint8 = (best_mask.astype(np.uint8) * 255)

    return blended, mask_uint8, confidence

# -----------------------------
# Gradio app logic
# -----------------------------
def segment_image(image, model_choice):
    """
    image: RGB numpy array from Gradio
    model_choice: "U-Net", "Mask R-CNN", or "Both"
    """
    if image is None:
        return None, None, "Please upload an MRI image."

    if model_choice == "U-Net":
        overlay, mask, conf = run_unet(image)
        text = f"U-Net confidence (max prob): {conf:.3f}"
        return overlay, mask, text

    elif model_choice == "Mask R-CNN":
        overlay, mask, conf = run_rcnn(image)
        text = f"Mask R-CNN confidence (top detection score): {conf:.3f}"
        return overlay, mask, text

    else:  # Both
        overlay_u, mask_u, conf_u = run_unet(image)
        overlay_r, mask_r, conf_r = run_rcnn(image)

        # Side-by-side overlays and masks
        overlay_combined = np.concatenate([overlay_u, overlay_r], axis=1)
        mask_u_rgb = cv2.cvtColor(mask_u, cv2.COLOR_GRAY2RGB)
        mask_r_rgb = cv2.cvtColor(mask_r, cv2.COLOR_GRAY2RGB)
        mask_combined = np.concatenate([mask_u_rgb, mask_r_rgb], axis=1)

        text = (
            f"U-Net confidence: {conf_u:.3f} | "
            f"Mask R-CNN confidence: {conf_r:.3f}"
        )
        return overlay_combined, mask_combined, text

# -----------------------------
# Build Gradio UI
# -----------------------------
with gr.Blocks(title="Brain Tumor Segmentation: U-Net vs Mask R-CNN") as demo:
    gr.Markdown(
        """
        # Brain Tumor Segmentation Demo
        Upload a T1-weighted brain MRI slice and choose which model to use.

        - **U-Net** → fast pixel-wise segmentation
        - **Mask R-CNN** → instance-level segmentation
        - **Both** → side-by-side comparison
        """
    )

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(
                label="Upload MRI slice (T1)",
                type="numpy"
            )
            model_choice = gr.Radio(
                ["U-Net", "Mask R-CNN", "Both"],
                value="U-Net",
                label="Select Model"
            )
            run_btn = gr.Button("Run Segmentation")

        with gr.Column():
            overlay_out = gr.Image(label="Segmentation Overlay")
            mask_out = gr.Image(label="Predicted Mask")
            conf_out = gr.Label(label="Model Confidence")

    run_btn.click(
        fn=segment_image,
        inputs=[img_input, model_choice],
        outputs=[overlay_out, mask_out, conf_out]
    )

if __name__ == "__main__":
    demo.launch(share=True)


Writing src/brain_tumor_demo_app.py


FileNotFoundError: [Errno 2] No such file or directory: 'src/brain_tumor_demo_app.py'